# SpamBase Long-Run Alpha Tracking (Single Configuration)

<a href="https://colab.research.google.com/github/CalculatedContent/xgboost2ww/blob/main/notebooks/SpamBase_LongRun_Alpha_Tracking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


This notebook tests whether the WeightWatcher α metric approaches 2 during long boosting sequences in XGBoost models trained on SpamBase. The experiment trains a single model for a large number of boosting rounds and periodically converts the model to WeightWatcher matrices using xgboost2ww. Alpha is measured using WeightWatcher to see if model quality improves as α approaches the critical HTSR regime.

## 1) Bootstrap environment

In [ ]:
%pip install -q xgboost weightwatcher scikit-learn pandas matplotlib seaborn scipy feather-format
%pip install -q git+https://github.com/CalculatedContent/xgboost2ww.git

## 2) Imports

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

import xgboost as xgb
import weightwatcher as ww
from xgboost2ww import convert

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42

## 3) Load SpamBase dataset

In [ ]:
spambase = fetch_openml(name='spambase', version=1, as_frame=True)
X_df = spambase.data.copy()
y = (spambase.target.astype(str) == '1').astype(int)

print(f'X shape: {X_df.shape}')
print(f'y mean (spam prevalence): {y.mean():.3f}')
X_df.head()

## 4) Train/test split (80/20) and scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_df,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Train:', X_train_scaled.shape, 'Test:', X_test_scaled.shape)

## 5) XGBoost DMatrix objects

In [ ]:
dtrain = xgb.DMatrix(X_train_scaled, label=y_train)
dtest = xgb.DMatrix(X_test_scaled, label=y_test)

## 6) Long-run single-configuration setup

In [ ]:
params = {
    'max_depth': 6,
    'eta': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 1,
    'reg_lambda': 1.0,
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'seed': RANDOM_STATE,
}

TOTAL_ROUNDS = 3000
CHUNK_SIZE = 50
N_STEPS = TOTAL_ROUNDS // CHUNK_SIZE
CHECKPOINT_EVERY_STEPS = 2

in_colab = False
try:
    from google.colab import drive
    in_colab = True
except ImportError:
    drive = None

if in_colab:
    drive.mount('/content/drive', force_remount=False)
    project_root = Path('/content/drive/MyDrive/xgboost2ww_runs/spambase_longrun_alpha_tracking')
    print(f'Google Drive enabled: {project_root}')
else:
    project_root = Path('.')
    print('Google Drive unavailable; using local notebook storage.')

results_dir = project_root / 'results'
models_dir = project_root / 'models'
results_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

results_path_csv = results_dir / 'spambase_longrun_alpha_tracking.csv'
results_path_feather = results_dir / 'spambase_longrun_alpha_tracking.feather'
latest_model_path = models_dir / 'spambase_longrun_latest.json'

print(f'TOTAL_ROUNDS={TOTAL_ROUNDS}, CHUNK_SIZE={CHUNK_SIZE}, N_STEPS={N_STEPS}')
print(f'CHECKPOINT_EVERY_STEPS={CHECKPOINT_EVERY_STEPS}')
print(f'Results CSV path: {results_path_csv}')



## 7) Incremental long-run training + WeightWatcher alpha tracking (W2, W7, W8)

In [ ]:
def compute_alpha_for_W(bst, W_name, current_round):
    layer = convert(
        bst,
        X_train_scaled,
        y_train,
        W=W_name,
        return_type='torch',
        nfolds=5,
        t_points=min(current_round, 160),
        random_state=RANDOM_STATE,
        train_params=params,
        num_boost_round=current_round,
    )
    watcher = ww.WeightWatcher(model=layer)
    df = watcher.analyze(randomize=True, detX=True)
    return float(df['alpha'].iloc[0]), df

rows = []
bst = None

for step in range(1, N_STEPS + 1):
    bst = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=CHUNK_SIZE,
        xgb_model=bst,
        verbose_eval=False,
    )

    current_round = step * CHUNK_SIZE

    y_prob = bst.predict(dtest)
    y_pred = (y_prob >= 0.5).astype(int)
    test_acc = accuracy_score(y_test, y_pred)

    alpha_W2, _ = compute_alpha_for_W(bst, 'W2', current_round)
    alpha_W7, _ = compute_alpha_for_W(bst, 'W7', current_round)
    alpha_W8, _ = compute_alpha_for_W(bst, 'W8', current_round)

    row = {
        'boosting_round': current_round,
        'test_accuracy': test_acc,
        'alpha_W2': alpha_W2,
        'alpha_W7': alpha_W7,
        'alpha_W8': alpha_W8,
    }
    rows.append(row)

    results_df = pd.DataFrame(rows)

    should_checkpoint = (step % CHECKPOINT_EVERY_STEPS == 0) or (step == N_STEPS)
    if should_checkpoint:
        results_df.to_csv(results_path_csv, index=False)
        try:
            results_df.to_feather(results_path_feather)
        except Exception as e:
            print(f'[WARN] Feather save skipped: {e}')

        model_path = models_dir / f'spambase_longrun_round_{current_round}.json'
        bst.save_model(str(model_path))
        bst.save_model(str(latest_model_path))

        print(f'[CHECKPOINT] saved metrics -> {results_path_csv}')
        print(f'[CHECKPOINT] saved model   -> {model_path}')

    print(
        f'Round {current_round:4d} | acc={test_acc:.4f} | ' +
        f'alpha_W2={alpha_W2:.3f}, alpha_W7={alpha_W7:.3f}, alpha_W8={alpha_W8:.3f}'
    )

results_df.tail()



## 8) Plot dynamics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Plot 1: Accuracy vs boosting round
axes[0].plot(results_df['boosting_round'], results_df['test_accuracy'], marker='o')
axes[0].set_title('Test Accuracy vs Boosting Round')
axes[0].set_xlabel('Boosting round')
axes[0].set_ylabel('Test accuracy')

# Plot 2: Alpha vs boosting round
axes[1].plot(results_df['boosting_round'], results_df['alpha_W2'], marker='o', label='alpha_W2')
axes[1].plot(results_df['boosting_round'], results_df['alpha_W7'], marker='o', label='alpha_W7')
axes[1].plot(results_df['boosting_round'], results_df['alpha_W8'], marker='o', label='alpha_W8')
axes[1].axhline(2.0, linestyle='--', color='black', alpha=0.7, label='alpha=2')
axes[1].set_title('Alpha vs Boosting Round')
axes[1].set_xlabel('Boosting round')
axes[1].set_ylabel('Alpha')
axes[1].legend()

# Plot 3: Accuracy vs alpha
axes[2].scatter(results_df['alpha_W2'], results_df['test_accuracy'], label='W2', alpha=0.8)
axes[2].scatter(results_df['alpha_W7'], results_df['test_accuracy'], label='W7', alpha=0.8)
axes[2].scatter(results_df['alpha_W8'], results_df['test_accuracy'], label='W8', alpha=0.8)
axes[2].axvline(2.0, linestyle='--', color='black', alpha=0.7)
axes[2].set_title('Test Accuracy vs Alpha')
axes[2].set_xlabel('Alpha')
axes[2].set_ylabel('Test accuracy')
axes[2].legend()

plt.tight_layout()
plt.show()

## 9) HTSR hypothesis check and final summary table

In [ ]:
summary_cols = ['boosting_round', 'test_accuracy', 'alpha_W2', 'alpha_W7', 'alpha_W8']

best_idx = results_df['test_accuracy'].idxmax()
best_row = results_df.loc[best_idx, summary_cols]
print('Best-accuracy row:')
display(best_row.to_frame().T)

near_best = results_df.sort_values('test_accuracy', ascending=False).head(10).copy()
near_best = near_best[summary_cols].sort_values('boosting_round')
print('Alpha values near peak accuracy (top-10 rounds):')
display(near_best)

print('\nStart -> end alpha trend:')
for col in ['alpha_W2', 'alpha_W7', 'alpha_W8']:
    print(f'{col}: {results_df[col].iloc[0]:.3f} -> {results_df[col].iloc[-1]:.3f}')

print('\nAccuracy/alpha correlations:')
for col in ['alpha_W2', 'alpha_W7', 'alpha_W8']:
    corr = results_df['test_accuracy'].corr(results_df[col])
    print(f'corr(test_accuracy, {col}) = {corr:.4f}')

### Interpretation guide
- If alpha is high early, then declines as rounds increase, and values near peak accuracy are close to ~2, that supports the HTSR-style hypothesis.
- If alpha does not decline or peak accuracy occurs far from alpha~2, then this run does not support that hypothesis under this operator and training path.
- This is intentionally a **single long-run configuration** (no hyperparameter sweep), so the result isolates trajectory effects.